In [ ]:
!pip install ultralytics opencv-python-headless

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

In [ ]:
yolo_model = YOLO("yolov8n.pt")

TARGET_CLASSES = [2, 3, 5, 7]  # car, motorcycle, bus, truck

In [ ]:
def get_image_edges(image_frame):
    grayscale = cv2.cvtColor(image_frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(grayscale, (5, 5), 0)
    edges = cv2.Canny(blurred, 50, 150)
    return edges


def mask_roi(edge_frame):
    height, width = edge_frame.shape
    blank_mask = np.zeros_like(edge_frame)

    roi_corners = np.array([[
        (100, height),
        (width//2 - 80, int(height*0.6)),
        (width//2 + 80, int(height*0.6)),
        (width - 100, height)
    ]], np.int32)

    cv2.fillPoly(blank_mask, roi_corners, 255)
    masked_image = cv2.bitwise_and(edge_frame, blank_mask)
    return masked_image, roi_corners


def calculate_lane_lines(hough_lines):
    left_lane, right_lane = [], []
    if hough_lines is None:
        return None, None

    for line in hough_lines:
        x1, y1, x2, y2 = line[0]
        if x1 == x2:
            continue
        m = (y2 - y1) / (x2 - x1)
        b = y1 - m * x1
        if m < -0.5:
            left_lane.append((m, b))
        elif m > 0.5:
            right_lane.append((m, b))

    left_avg = np.mean(left_lane, axis=0) if left_lane else None
    right_avg = np.mean(right_lane, axis=0) if right_lane else None
    return left_avg, right_avg


def generate_line_coordinates(frame, line_params):
    if line_params is None:
        return None
    height = frame.shape[0]
    slope, intercept = line_params
    y_bottom = height
    y_top = int(height * 0.6)
    x_bottom = int((y_bottom - intercept) / slope)
    x_top = int((y_top - intercept) / slope)
    return (x_bottom, y_bottom, x_top, y_top)

In [ ]:
tracker_dict = {}
total_vehicles = 0
current_id = 0

def check_inside_polygon(x, y, poly):
    return cv2.pointPolygonTest(poly, (x, y), False) >= 0

In [ ]:
video_input = "/content/project_video.mp4"
video_cap = cv2.VideoCapture(video_input)

width = int(video_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(video_cap.get(cv2.CAP_PROP_FPS))
if fps == 0:
    fps = 25

video_out = cv2.VideoWriter(
    "/content/adas_output.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

while video_cap.isOpened():
    success, curr_frame = video_cap.read()
    if not success:
        break
    
    frame_edges = get_image_edges(curr_frame)
    roi_edges, lane_polygon = mask_roi(frame_edges)

    detected_lines = cv2.HoughLinesP(
        roi_edges, 1, np.pi/180, 100,
        minLineLength=60, maxLineGap=150
    )

    l_line_params, r_line_params = calculate_lane_lines(detected_lines)
    left_coords = generate_line_coordinates(curr_frame, l_line_params)
    right_coords = generate_line_coordinates(curr_frame, r_line_params)

    center_of_lane = None
    if left_coords and right_coords:
        cv2.line(curr_frame, left_coords[:2], left_coords[2:], (0,255,0), 5)
        cv2.line(curr_frame, right_coords[:2], right_coords[2:], (0,255,0), 5)
        center_of_lane = (left_coords[0] + right_coords[0]) // 2
        cv2.line(curr_frame, (center_of_lane, height), (center_of_lane, int(height*0.6)), (255,0,0), 3)

    cv2.polylines(curr_frame, lane_polygon, True, (255,255,0), 2)

    yolo_results = yolo_model(curr_frame, conf=0.25, verbose=False)[0]

    risk_of_collision = False

    for bbox in yolo_results.boxes:
        class_id = int(bbox.cls.item())
        if class_id not in TARGET_CLASSES:
            continue

        x1, y1, x2, y2 = map(int, bbox.xyxy[0])
        center_x, center_y = (x1 + x2)//2, (y1 + y2)//2
        box_area = (x2 - x1) * (y2 - y1)
        conf_score = float(bbox.conf.item())

        box_label = f"{yolo_model.names[class_id]} {conf_score:.2f}"

        cv2.rectangle(curr_frame, (x1,y1), (x2,y2), (255,255,0), 1)
        cv2.putText(curr_frame, box_label, (x1,y1-8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 2)

        if not check_inside_polygon(center_x, center_y, lane_polygon[0]):
            continue

        found_match = False
        for v_id, (prev_x, prev_y) in tracker_dict.items():
            if abs(center_x - prev_x) < 40 and abs(center_y - prev_y) < 40:
                tracker_dict[v_id] = (center_x, center_y)
                found_match = True
                break

        if not found_match:
            tracker_dict[current_id] = (center_x, center_y)
            total_vehicles += 1
            current_id += 1

        if center_y > 0.75 * height or box_area > 40000:
            risk_of_collision = True
            cv2.rectangle(curr_frame, (x1,y1), (x2,y2), (0,0,255), 3)

    if risk_of_collision:
        cv2.putText(curr_frame, "CRASH WARNING!",
                    (50, 100),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.1, (0,0,255), 3)

    if center_of_lane is not None:
        vehicle_center = width // 2
        if abs(vehicle_center - center_of_lane) > 50:
            cv2.putText(curr_frame, "LANE DEPARTURE!",
                        (50, 150),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        1.1, (0,0,255), 3)

    cv2.putText(curr_frame, f"Total Vehicles: {total_vehicles}",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1, (0,255,255), 2)

    video_out.write(curr_frame)

video_cap.release()
video_out.release()

print("Finished saving ADAS output video.")

In [ ]:
from google.colab import files
files.download("/content/adas_output.mp4")